In [1]:
# Skipped in CI: Colab/bootstrap dependency install cell.


import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
# Case Study: Event-Synchronized Analysis

This tutorial shows a real-world workflow: matching events from two different logs, applying clock corrections, and generating event-by-event summary plots.

The event windows are still represented by `gwpy.segments.Segment`, while gwexpy provides the table-management and `apply()`-driven batch workflow through `SegmentTable`. The division between GWpy base classes and gwexpy extensions is the same as in [SegmentTable: Basics](intro_segment_table.ipynb).


In [2]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    import pandas as pd
    from gwpy.segments import Segment

    from gwexpy.table import SegmentTable

    # 1. Simulate finding events
    events_df = pd.DataFrame({
    "gps": [1234567890.1, 1234567895.5, 1234567900.2],
    "snr": [15, 8, 22]
    })

    # 2. Create SegmentTable from events
    # Define a 4-second window around each GPS time
    segs = [Segment(t-2, t+2) for t in events_df["gps"]]
    st = SegmentTable.from_segments(segs, snr=events_df["snr"])
    st


## Batch Plot Generation

We can use `apply()` to generate plots for each event and save them, storing the paths back in the table.

In [3]:
import os

os.makedirs("outputs", exist_ok=True)

def generate_event_plot(row):
    # Simulate plotting
    path = f"outputs/event_{row.index}.png"
    # plot = row["raw"].plot() -> plot.save(path)
    with open(path, "w") as f: f.write("Dummy PNG")
    return {"plot_path": path}

st_plots = st.apply(generate_event_plot)
st_plots.display()

,span,snr,plot_path
0,"(1234567888.1, 1234567892.1)",15,outputs/event_0.png
1,"(1234567893.5, 1234567897.5)",8,outputs/event_1.png
2,"(1234567898.2, 1234567902.2)",22,outputs/event_2.png


## Result Summary

Finally, we can filter the table and export results.

In [4]:
st_best = st_plots.select(mask=st_plots.to_pandas()["snr"] > 10)
st_best.to_pandas()

,span,snr,plot_path
0,"(1234567888.1, 1234567892.1)",15,outputs/event_0.png
1,"(1234567898.2, 1234567902.2)",22,outputs/event_2.png
